# CMS Medical Care Ratings vs Enrollee Experience

## Exploratory Analysis

**Main question:** Are stronger CMS Medical Care ratings associated with better enrollee-reported health plan experiences?

This notebook continues the project after data intake, cleaning, and merging were completed in Notebook 01. It uses the 308-row analysis-ready dataset created from the CMS Quality Rating System and QHP Enrollee Survey files.

**Primary variables**
- `S1_rate`: Medical Care star rating
- `S1_score`: Medical Care continuous score
- `RATE_HP_SCORE_ADJ`: Adjusted enrollee-reported health plan rating


## 1. Load the analysis-ready dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

project_root = Path("..")
data_clean = project_root / "data_clean"
analysis_file = data_clean / "cms_qrs_enrollee_experience_clean.csv"

df = pd.read_csv(analysis_file)

print("Dataset shape:", df.shape)
display(df.head())

The analysis-ready file contains **308 reporting units and 22 columns**. This matches the reconciled row count used in MySQL and Tableau.

## 2. Review the primary analysis variables

In [ ]:
analysis_vars = [
    "S1_rate",
    "S1_score",
    "RATE_HP_SCORE_ADJ"
]

display(df[analysis_vars].dtypes)
display(df[analysis_vars].describe())

All three primary variables are numeric in the analysis-ready dataset, so they can be used directly for descriptive statistics, grouped comparisons, and correlation analysis.

## 3. Distribution of enrollee-reported health plan experience

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["RATE_HP_SCORE_ADJ"], bins=15)
plt.xlabel("Adjusted Health Plan Rating Score")
plt.ylabel("Number of Reporting Units")
plt.title("Distribution of Enrollee-Reported Health Plan Experience")
plt.show()

In [ ]:
# Measure skewness of the target variable
experience_skew = df["RATE_HP_SCORE_ADJ"].skew()

print("Health Plan Rating skewness:", round(experience_skew, 3))

**Result:** skewness = **-0.151**.

The adjusted health plan rating distribution is close to symmetric, with only slight negative skew. This supports using standard summary statistics such as the mean while still reviewing the full distribution.

## 4. Enrollee experience by Medical Care star rating

In [ ]:
# Summarize enrollee experience by Medical Care star rating
rating_summary = (
    df.groupby("S1_rate")["RATE_HP_SCORE_ADJ"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(2)
)

display(rating_summary)

In [ ]:
# Average Health Plan Rating by Medical Care star level
mean_experience = (
    df.groupby("S1_rate")["RATE_HP_SCORE_ADJ"]
    .mean()
    .round(2)
)

display(mean_experience)

**Average enrollee experience by Medical Care rating**

- 1 star: **73.76**
- 2 stars: **73.43**
- 3 stars: **71.08**
- 4 stars: **70.17**
- 5 stars: **73.45**

The averages do **not** increase steadily as Medical Care star ratings rise. The pattern is non-linear, which suggests that a stronger Medical Care rating does not automatically correspond with a better overall member experience.

## 5. Correlation between Medical Care score and enrollee experience

In [ ]:
# Pearson correlation
pearson_corr = df["S1_score"].corr(
    df["RATE_HP_SCORE_ADJ"],
    method="pearson"
)

# Spearman correlation
spearman_corr = df["S1_score"].corr(
    df["RATE_HP_SCORE_ADJ"],
    method="spearman"
)

print("Pearson correlation:", round(pearson_corr, 3))
print("Spearman correlation:", round(spearman_corr, 3))

### Correlation Findings

Medical Care Score showed a **weak negative relationship** with adjusted Health Plan Experience Score.

- Pearson correlation: **-0.132**
- Spearman correlation: **-0.159**

Both measures suggest that higher Medical Care performance was not associated with better enrollee-reported experience in this sample. The relationship was weak, meaning Medical Care performance alone explains little of the variation in member experience.

This does **not** imply that stronger Medical Care causes worse member experience. The analysis is observational and the two measures capture different dimensions of health plan performance.

## 6. Key findings

1. Enrollee experience did not improve consistently as Medical Care star ratings increased.
2. The continuous Medical Care score had only a weak negative association with enrollee experience.
3. These results support treating Medical Care quality and member experience as related but distinct dimensions of health plan performance.
4. The analysis was continued in MySQL for reporting queries and in Tableau for the final dashboard and visualization layer.

## 7. Workflow handoff

**Python → MySQL → Tableau**

- Python: descriptive analysis, distribution review, and correlation analysis
- MySQL: reporting queries, row-count reconciliation, grouped summaries, and mismatch identification
- Tableau: final bar chart, scatterplot, mismatch ranking, and dashboard
